# Chorin projection: a VAM channel

Goal: Split the vertically averaged model `VAM(level=1)` into predictor, pressure and correction, and march a 2-D channel with the Chorin solver on NumPy and on JAX.

Prerequisite: [Quick start: a 2-D bump on NumPy and JAX](../swe/bump_2d.ipynb)

## 1. Model and split

A model with a non-hydrostatic pressure is solved by projection. `chorin_split` returns the three systems the solver needs: the predictor `SM_pred`, the pressure system `SM_press` and the correction `SM_corr`.

In [1]:
import numpy as np
import jax

jax.config.update("jax_enable_x64", True)

from zoomy_core.mesh import LSQMesh
from zoomy_core.model.boundary_conditions import Dirichlet, Extrapolation
from zoomy_core.model.initial_conditions import Constant
from zoomy_core.model.models.closures import Newtonian, StressFree
from zoomy_core.model.models.vam import VAM
from zoomy_core.symsolver.settings import LinearSolver
from zoomy_core.symsolver.strategies.chorin import predictor_nsm
from zoomy_core.systemmodel.system_model import _DT_SYMBOL, SystemModel

bcs = ([Dirichlet("left", on=f, value=v)
        for f, v in (("q_x_0", 0.11197), ("q_x_1", 0.0), ("r_0", 0.0), ("r_1", 0.0))]
       + [Dirichlet("right", on=f"P_{i}", value=0.0) for i in range(2)]
       + [Extrapolation(tag=t) for t in ("bottom", "top")])
model = VAM(level=1, dimension=3, boundary_conditions=bcs, closures=[Newtonian(), StressFree()])
sm = SystemModel.from_model(model)
sm.initial_conditions = Constant(constants=lambda n: np.zeros(n))
sm.aux_initial_conditions = Constant(constants=lambda n: np.zeros(n))

split = model.chorin_split(_DT_SYMBOL, system_model=sm)
press, corr = split.SM_press, split.SM_corr
pred = predictor_nsm(split.SM_pred, press, corr)     # the predictor's numerics
names = [str(s) for s in pred.state]
print(names)

['b', 'h', 'q_x_0', 'q_x_1', 'q_y_0', 'q_y_1', 'r_0', 'r_1', 'P_0', 'P_1']


## 2. NumPy

The initial state is a reservoir upstream of a bump, shallow water downstream.

In [2]:
from zoomy_core.symsolver.numpy_backend import SymbolicChorinSolverNumpy

mesh = LSQMesh.create_2d(domain=(-1.5, 1.5, 0.0, 0.3), nx=60, ny=6)
dt = 2.0 ** -10


def initial_state(solver, Q):
    x = np.asarray(solver.mesh_np.cell_centers)[0, :Q.shape[1]]
    Q0 = np.zeros(Q.shape)
    b = 0.20 * np.exp(-x ** 2 / (2 * 0.20 ** 2))
    Q0[names.index("b")] = b
    Q0[names.index("h")] = np.where(x < 0.0, 0.34 - b, 0.015)
    return Q0


numpy_solver = SymbolicChorinSolverNumpy(time_end=8 * dt, dt=dt, time_order=1,
                                         pressure_linear_solver=LinearSolver.gmres)
Q, Qaux = numpy_solver.setup(mesh, pred, press, corr)
Q0 = initial_state(numpy_solver, Q)
Qaux = numpy_solver.initial_aux(Q0, Qaux)
time, Q, Qaux = numpy_solver.run(Q0, Qaux)

h_numpy = np.asarray(Q)[names.index("h")]
print(f"t = {time:.6f}, cells = {h_numpy.size}, h in [{h_numpy.min():.5f}, {h_numpy.max():.5f}]")

t = 0.007812, cells = 360, h in [0.01500, 0.34276]


## 3. JAX

The same three systems, the JAX solver.

In [3]:
from zoomy_jax.symsolver.solver import SymbolicChorinSolverJax

jax_solver = SymbolicChorinSolverJax(time_end=8 * dt, dt=dt, time_order=1,
                                     pressure_linear_solver=LinearSolver.gmres)
Q, Qaux = jax_solver.setup(mesh, pred, press, corr)
Q0 = initial_state(jax_solver, Q)
Qaux = jax_solver.initial_aux(jax.numpy.asarray(Q0), Qaux)
time, Q, Qaux = jax_solver.run(jax.numpy.asarray(Q0), Qaux)

h_jax = np.asarray(Q)[names.index("h")]
print(f"t = {float(time):.6f}, cells = {h_jax.size}, h in [{h_jax.min():.5f}, {h_jax.max():.5f}]")
print(f"max |h_jax - h_numpy| = {np.abs(h_jax - h_numpy).max():.1e}")

t = 0.007812, cells = 360, h in [0.01500, 0.34276]
max |h_jax - h_numpy| = 9.5e-15


In [4]:
assert np.abs(h_jax - h_numpy).max() < 1e-6, "NumPy and JAX disagree"